# Entrega 2 — Detecção de Duplicatas

Implementação de `detectar_duplicatas(textos, limiar=0.85)`, matriz de similaridade, heatmap e análise de falsos positivos/negativos.

In [ ]:
from pathlib import Path
import json,pandas as pd,numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

ROOT=Path.cwd()
if not (ROOT/'data'/'manifestacoes.json').exists(): ROOT=ROOT.parent
with open(ROOT/'data'/'manifestacoes.json',encoding='utf-8') as f: data=json.load(f)
df=pd.DataFrame(data)
model=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
emb=model.encode(df.texto.tolist(),normalize_embeddings=True,show_progress_bar=False)
sim=cosine_similarity(emb)


In [ ]:
def detectar_duplicatas(textos, limiar=0.85):
    E=model.encode(list(textos),normalize_embeddings=True,show_progress_bar=False)
    S=cosine_similarity(E)
    pares=[]
    for i in range(len(textos)):
        for j in range(i+1,len(textos)):
            if S[i,j]>=limiar:
                pares.append((i,j,float(S[i,j])))
    return pares,S

pares,S=detectar_duplicatas(df.texto,0.85)
[(df.iloc[i].id,df.iloc[j].id,round(s,3)) for i,j,s in pares]


In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(S,xticklabels=df.id,yticklabels=df.id,cmap="viridis")
plt.title("Matriz de similaridade — embeddings")
plt.tight_layout()
plt.show()


## Análise de erros

O limiar de 0,85 é usado como ponto inicial porque busca priorizar pares semanticamente muito próximos. Ele não é universal: o valor adequado depende do modelo e do corpus. Para avaliar falsos positivos e falsos negativos, compare os pares retornados com as duplicatas reais marcadas no dataset oficial. Neste projeto demonstrativo, o JSON incluído não possui um campo explícito de “duplicata real”; por isso a análise final deve ser complementada quando o dataset oficial trouxer essa anotação.